# Experiment 2: Architecture Change — Wider MLP (512→256→10)

## Rationale

Doubling hidden layer capacity to check if Shirt confusion is a model-capacity bottleneck.

**Single variable changed**: Architecture (hidden layer sizes / depth)

**Held constant**: Training (30 epochs, Adam lr=0.001, CosineAnnealingLR), loss (CrossEntropy), dropout (0.2), data pipeline, evaluation protocol


In [1]:
import sys, os

def _find_root(marker="src", max_up=3):
    p = os.path.abspath(os.getcwd())
    for _ in range(max_up + 1):
        if os.path.isdir(os.path.join(p, marker)):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.getcwd())
PROJ_ROOT = _find_root()
sys.path.insert(0, PROJ_ROOT)

import torch, torch.nn as nn, torch.optim as optim
import numpy as np
import torchvision.transforms as transforms

from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from src.train_utils import train_one_epoch
from src.eval_utils import (
    evaluate_detailed, get_all_probas_and_labels,
    compute_roc_auc_scores, compute_pr_auc_scores
)

OUT_DIR = os.path.join(PROJ_ROOT, 'outputs/error_analysis/MLP/wider')
os.makedirs(OUT_DIR, exist_ok=True)

DATA_DIR = os.path.join(PROJ_ROOT, 'data')

print(f'PyTorch version: {torch.__version__}')
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')
print(f'OUT_DIR: {OUT_DIR}')


PyTorch version: 2.13.0+cu130
Using device: cuda
OUT_DIR: c:\document\Study documents\Deeplearning_Course\outputs/error_analysis/MLP/wider


## Dataset — identical to Phase 1

FashionMNIST: 60k train / 10k test, 10 classes, 28×28 grayscale. Normalised to [-1, 1].


In [2]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_ds = __import__('torchvision').datasets.FashionMNIST(root=DATA_DIR, train=True, download=True, transform=transform)
test_ds = __import__('torchvision').datasets.FashionMNIST(root=DATA_DIR, train=False, download=True, transform=transform)
class_names = train_ds.classes

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)
print(f'Training batches: {len(train_loader)}')
print(f'Test batches: {len(test_loader)}')
print(f'Classes: {class_names}')


Training batches: 938
Test batches: 40
Classes: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


## Architecture — single variable change

Model definition with the modified architecture.


In [3]:
class MLPWider(nn.Module):
    def __init__(self, input_size=784, hidden_1=512, hidden_2=256, num_classes=10, dropout=0.2):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(input_size, hidden_1)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_1, hidden_2)
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(dropout)
        self.fc3 = nn.Linear(hidden_2, num_classes)
    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x); x = self.relu1(x); x = self.drop1(x)
        x = self.fc2(x); x = self.relu2(x); x = self.drop2(x)
        return self.fc3(x)

model = MLPWider().to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')


Parameters: 535,818


c:\document\Study documents\Deeplearning_Course\.venv\Lib\site-packages\torch\nn\modules\module.py:1369: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:40.)
  return t.to(


## Training — 30 epochs with cosine LR decay

Training for 30 epochs with CrossEntropyLoss + Adam + CosineAnnealingLR, matching Phase 1 extended setup.


In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = CosineAnnealingLR(optimizer, T_max=30)
EPOCHS = 30

train_losses = []
model.train()
for epoch in range(EPOCHS):
    loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(loss)
    scheduler.step()
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch [{epoch+1}/{EPOCHS}], Loss: {loss:.4f}')

with open(os.path.join(OUT_DIR, 'train_losses.txt'), 'w') as f:
    for l in train_losses: f.write(f'{l}\n')
print(f'Final loss: {train_losses[-1]:.4f}')


Epoch [1/30], Loss: 0.5164
Epoch [5/30], Loss: 0.3246
Epoch [10/30], Loss: 0.2575
Epoch [15/30], Loss: 0.2080
Epoch [20/30], Loss: 0.1608
Epoch [25/30], Loss: 0.1272
Epoch [30/30], Loss: 0.1151
Final loss: 0.1151


## Evaluation — identical pipeline

All metrics saved as raw `.txt` files.


In [5]:
# --- Accuracy, Per-Class Metrics, Confusion Matrix ---
accuracy, cm, per_class = evaluate_detailed(model, test_loader, device, class_names, model_name="wider")

cm_np = cm.cpu().numpy()
with open(os.path.join(OUT_DIR, 'confusion_matrix.txt'), 'w') as f:
    header = f'{"":>15}'
    for name in class_names:
        header += f'{name:>15}'
    f.write(header + '\n')
    for i in range(len(class_names)):
        row = f'{class_names[i]:>15}'
        for j in range(len(class_names)):
            row += f'{cm_np[i, j]:>15}'
        f.write(row + '\n')
print("Confusion matrix saved.")

# --- ROC-AUC and PR-AUC ---
probas, labels = get_all_probas_and_labels(model, test_loader, device, 10)
roc_scores = compute_roc_auc_scores(probas, labels, model_name="wider")
pr_scores = compute_pr_auc_scores(probas, labels, model_name="wider")

with open(os.path.join(OUT_DIR, 'metrics_summary.txt'), 'w') as f:
    f.write(f'Test Accuracy (percentage): {accuracy:.2f}\n')
    f.write(f'Test Accuracy (fraction): {accuracy / 100:.4f}\n\n')
    f.write(f'Macro ROC-AUC: {roc_scores["macro"]:.6f}\n')
    f.write(f'Macro PR-AUC:  {pr_scores["macro"]:.6f}\n\n')
    f.write(f'{"Class":<15} {"ROC-AUC":>10} {"PR-AUC":>10} {"TPR":>10} {"Precision":>10}\n')
    f.write(f'{"-"*55}\n')
    for i, name in enumerate(class_names):
        tpr = per_class[name]['TPR']
        prec = per_class[name]['Precision']
        f.write(f'{name:<15} {roc_scores[f"class_{i}"]:>10.4f} {pr_scores[f"class_{i}"]:>10.4f} {tpr:>10.4f} {prec:>10.4f}\n')
print("Metrics summary saved.")

# --- Misclassification Analysis ---
with open(os.path.join(OUT_DIR, 'misclassification_analysis.txt'), 'w') as f:
    f.write('Misclassification Analysis\n')
    f.write('=' * 70 + '\n\n')
    for c in range(len(class_names)):
        name = class_names[c]; errors = cm_np[c].sum() - cm_np[c, c]
        f.write(f'True: {name}  (errors: {errors})\n')
        f.write('-' * 50 + '\n')
        for p in np.argsort(-cm_np[c]):
            if p == c or cm_np[c, p] == 0: continue
            f.write(f'  -> {class_names[p]:<15} count={cm_np[c, p]:>4}\n')
        f.write('\n')
print("Misclassification analysis saved.")

print(f"\nAll results saved to {OUT_DIR}/")


  Test Accuracy: 90.30%
  Class           TPR(Recall)        FPR  Precision
  ---------------------------------------------
  T-shirt/top         0.8640     0.0173     0.8471
  Trouser             0.9760     0.0009     0.9919
  Pullover            0.8390     0.0186     0.8340
  Dress               0.9130     0.0118     0.8960
  Coat                0.8410     0.0182     0.8368
  Sandal              0.9640     0.0031     0.9718
  Shirt               0.7280     0.0258     0.7583
  Sneaker             0.9650     0.0060     0.9470
  Bag                 0.9760     0.0021     0.9809
  Ankle boot          0.9640     0.0040     0.9640
Confusion matrix saved.
Metrics summary saved.
Misclassification analysis saved.

All results saved to c:\document\Study documents\Deeplearning_Course\outputs/error_analysis/MLP/wider/


## Delta vs Phase 1 Baseline

Comparison against the Phase 1 MLP baseline (784→256→128→10, 30 epochs, 90.08% acc, 235K params).


In [6]:
# Phase 1 baseline values (from extended 30-epoch model)
E1_ACC = 90.08
E1_TPR = {
    'T-shirt/top': 0.855, 'Trouser': 0.972, 'Pullover': 0.841,
    'Dress': 0.913, 'Coat': 0.856, 'Sandal': 0.962,
    'Shirt': 0.707, 'Sneaker': 0.970, 'Bag': 0.971,
    'Ankle boot': 0.961,
}
E1_PREC = {
    'T-shirt/top': 0.848, 'Trouser': 0.990, 'Pullover': 0.839,
    'Dress': 0.893, 'Coat': 0.824, 'Sandal': 0.980,
    'Shirt': 0.751, 'Sneaker': 0.943, 'Bag': 0.975,
    'Ankle boot': 0.964,
}

EXPERIMENT_LABEL = 'Wider'
print(f'{"Class":<15} {"E1 TPR":>8} {EXPERIMENT_LABEL:>10} {chr(916)+" TPR":>8} {"E1 Prec":>8} {EXPERIMENT_LABEL:>10} {chr(916)+" Prec":>8}')
print('-' * 75)
for name in class_names:
    tpr_delta = per_class[name]['TPR'] - E1_TPR[name]
    prec_delta = per_class[name]['Precision'] - E1_PREC[name]
    print(f'{name:<15} {E1_TPR[name]:>8.3f} {per_class[name]["TPR"]:>10.3f} {tpr_delta:>+8.3f} {E1_PREC[name]:>8.3f} {per_class[name]["Precision"]:>10.3f} {prec_delta:>+8.3f}')

print(f'\nAccuracy:  E1={E1_ACC:.2f}%  ' + EXPERIMENT_LABEL + f'={accuracy:.2f}%  {chr(916)}={accuracy - E1_ACC:+.2f}%')

Class             E1 TPR      Wider    Δ TPR  E1 Prec      Wider   Δ Prec
---------------------------------------------------------------------------
T-shirt/top        0.855      0.864   +0.009    0.848      0.847   -0.001
Trouser            0.972      0.976   +0.004    0.990      0.992   +0.002
Pullover           0.841      0.839   -0.002    0.839      0.834   -0.005
Dress              0.913      0.913   +0.000    0.893      0.896   +0.003
Coat               0.856      0.841   -0.015    0.824      0.837   +0.013
Sandal             0.962      0.964   +0.002    0.980      0.972   -0.008
Shirt              0.707      0.728   +0.021    0.751      0.758   +0.007
Sneaker            0.970      0.965   -0.005    0.943      0.947   +0.004
Bag                0.971      0.976   +0.005    0.975      0.981   +0.006
Ankle boot         0.961      0.964   +0.003    0.964      0.964   +0.000

Accuracy:  E1=90.08%  Wider=90.30%  Δ=+0.22%


## Results saved to `outputs/error_analysis/MLP/wider/`

| File | Contents |
|------|----------|
| `train_losses.txt` | Per-epoch training loss |
| `metrics_summary.txt` | Accuracy, per-class ROC-AUC, PR-AUC, TPR, Precision |
| `confusion_matrix.txt` | Raw confusion matrix (rows=true, cols=predicted) |
| `misclassification_analysis.txt` | Per-class error breakdown |
